<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/notebooks/predict_mortality_using_mimic_iv_baseline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Xây dựng mô hình dự đoán tỷ lệ tử vong của ICU từ MIMIC-IV (BaseLine1)**

**Mục tiêu:**

Notebook này thực hiện xây dựng một mô hình dự đoán tỷ lệ tử vong từ bảng MIMIC bằng các mô hình học máy. Nhằm tạo ra một chuẩn dữ liệu để so sánh với các kỹ thuật khác trong tương lai (nếu có)

**Bộ dữ liệu**

MIMIC-IV là cơ sở dữ liệu lớn gồm các bảng về thông tin bệnh nhân, lâm sàng, các thủ thuật, v...

Lý do chọn bộ dữ liệu này do tính phức tạp, sát với dữ liệu thực tế, việc xử lý dữ liệu này là một trong những bước khó khắn, ...

- 2 module chính được sử dụng trong notebook này chính là `hosp` và `icu`. Chi tiết các bảng lựa chọn sẽ được mô tả ở dưới


## **1. Nạp thư viện và dữ liệu cần thiết**

In [4]:
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

import os
from typing import List, Tuple, Dict, Optional

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Libraries loaded


In [5]:
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq

def convert_csv_to_parquet_batch(
    table_configs,
    input_dir,
    output_dir,
    folder,
    block_size=100_000_000,
    compression='snappy'
):
    """
    Chuyển đổi danh sách các bảng csv.gz sang Parquet phân theo module.
    """
    # Ghép folder vào đường dẫn output_dir
    target_output_dir = os.path.join(output_dir, folder)
    target_input_dir = os.path.join(input_dir, folder)

    os.makedirs(target_output_dir, exist_ok=True)
    read_options = pv.ReadOptions(block_size=block_size)

    print(f"ĐANG XỬ LÝ MODULE: [{folder.upper()}]")


    for filename, columns in table_configs.items():
        input_file_path = os.path.join(target_input_dir, filename)

        # Lưu kết quả vào đúng thư mục con của module đó
        output_filename = filename.replace('.csv.gz', '.parquet').replace('.csv', '.parquet')
        output_file_path = os.path.join(target_output_dir, output_filename)

        print(f"\n- Đang xử lý: {filename}")
        print(f"  + Nguồn: {input_file_path}")
        print(f"  + Đích:  {output_file_path}")

        if not os.path.exists(input_file_path):
            print(f"Cảnh báo: Không tìm thấy file '{input_file_path}'. Đang bỏ qua...")
            continue

        convert_options = pv.ConvertOptions(include_columns=columns) if columns else pv.ConvertOptions()

        try:
            reader = pv.open_csv(
                input_file_path,
                read_options=read_options,
                convert_options=convert_options
            )

            writer = None
            for i, batch in enumerate(reader):
                table = pa.Table.from_batches([batch])

                if writer is None:
                    writer = pq.ParquetWriter(output_file_path, table.schema, compression=compression)

                writer.write_table(table)
                print(f"   -> [{filename}] Đã ghi batch {i + 1}...")

            if writer:
                writer.close()
                print(f"Hoàn thành: {output_filename}")
            else:
                print(f"File rỗng hoặc không có dữ liệu.")

        except Exception as e:
            print(f"Lỗi trong quá trình xử lý file {filename}: {e}")

    print(f"\nĐã hoàn thành toàn bộ bảng trong module [{folder.upper()}]!")

In [125]:
# # Đường dẫn gốc đến dữ liệu nguồn và dữ liệu đích
# BASE_INPUT_DIR = '/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1'
# BASE_OUTPUT_DIR = '/content/drive/MyDrive/NCKH-DDU1231/outputs_parquet'

# BLOCK_SIZE = 100_000_000  # 100MB cho mỗi batch đọc
# COMPRESSION = 'snappy'

# # ĐỊNH NGHĨA CÁC MODULE VÀ BẢNG TƯƠNG ỨNG
# MODULE_CONFIGS = {
#     'icu': {
#         'chartevents.csv.gz': [
#             'subject_id', 'hadm_id', 'stay_id',
#             'charttime', 'itemid', 'valuenum',
#             'value', 'valueuom'
#         ],

#         'inputevents.csv.gz': None,
#         'outputevents.csv.gz': None,
#         'procedureevents.csv.gz': None,
#         'icustays.csv.gz': None,
#     },

#     'hosp': {
#         'patients.csv.gz': [
#             'subject_id', 'gender', 'anchor_age'
#         ],

#         'admissions.csv.gz': None,

#         'labevents.csv.gz': [
#             'subject_id', 'hadm_id',
#             'charttime', 'itemid', 'valuenum',
#             'value', 'valueuom'
#         ]
#     }
# }

In [124]:
# for folder_name, table_configs in MODULE_CONFIGS.items():
#         convert_csv_to_parquet_batch(
#             table_configs=table_configs,
#             input_dir=BASE_INPUT_DIR,
#             output_dir=BASE_OUTPUT_DIR,
#             folder=folder_name,
#             block_size=BLOCK_SIZE,
#             compression=COMPRESSION
#         )

# print("\nTẤT CẢ CÁC MODULE ĐÃ ĐƯỢC CHUYỂN ĐỔI THÀNH CÔNG!")

In [6]:
# Đường dẫn gốc tới thư mục chứa dữ liệu Parquet
DATA_DIR = '/content/drive/MyDrive/NCKH-DDU1231/outputs_parquet'

def load(folder, name, columns=None):
    """
    Nạp dữ liệu từ file Parquet.

    :param folder: Tên module (vd: 'icu', 'hosp', 'ed')
    :param name: Tên file (vd: 'chartevents.parquet' hoặc 'chartevents')
    :param columns: Danh sách các cột cần lấy (mặc định None - lấy tất cả).
                    Đọc cột chọn lọc với Parquet giúp tiết kiệm đáng kể RAM.
    """
    path = os.path.join(DATA_DIR, folder, name)

    if os.path.exists(path):
        # pd.read_parquet hỗ trợ đọc chỉ các cột cần thiết qua tham số columns
        df = pd.read_parquet(path, columns=columns)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY (Đường dẫn: {path})")
        return pd.DataFrame()

## **2. Tiền xử lý dữ liệu trước khi chia huấn luyện**

### **2.1. Xây dựng tập cohort**

Cohort được xây dựng bám sát theo các tiêu chí như sau:
- Phải là người trưởng thành (anchor_age > 18)
- Thời gian nằm ICU phải trên 24h (LOS >= 24)
- Chỉ lấy lần nhập ICU đầu tiên trong một lần nhập viện
- Loại các bệnh nhân đã tử vong, xuât hoặc chuyển viện trong vòng 24h đầu

In [7]:
def build_and_val_icu_cohort(patients_df: pd.DataFrame, admissions_df: pd.DataFrame, icustays_df: pd.DataFrame) -> pd.DataFrame:
  # Chuyển đổi định dạng mốc thời gian sang Datetime
  time_cols = {
      'icustays_df': ['intime', 'outtime'],
      'admissions_df': ['admittime', 'dischtime', 'edregtime', 'edouttime'],
      'patients_df' : []
  }

  for col in time_cols['icustays_df']:
        icustays_df[col] = pd.to_datetime(icustays_df[col])
  for col in time_cols['admissions_df']:
        admissions_df[col] = pd.to_datetime(admissions_df[col])
  # 1. Inner Join với patients
  cohort = icustays_df.merge(
  admissions_df[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
                        "admission_location", "deathtime",
                        "hospital_expire_flag"]],
          on=['subject_id', 'hadm_id'],
          how='inner'
      )
  # 2. Inner join với admission
  cohort = cohort.merge(
          patients_df[["subject_id", "gender", "anchor_age"]],
          on='subject_id',
          how='inner'
      )

  initial_count = len(cohort)
  print(f"Tổng số lượt ICU ban đầu: {initial_count:,}")

  # Áp dụng các tiêu chí lọc
  # ---------------------------------------------
  # Tiêu chí 1: Người trưởng thành (>= 18 tuổi)
  # ----------------------------------------------
  cohort = cohort[cohort["anchor_age"] >= 18]
  print(f"-> Sau khi lọc người trưởng thành (>=18t): {len(cohort):,} ca")

  # ---------------------------------------------
  # Tiêu chí 2: Lấy ca ICU đầu tiên của mỗi bệnh nhân (First ICU stay per patient)
  # Sắp xếp theo intime để chắc chắn lấy ca đầu tiên trong đời/lịch sử của bệnh nhân
  # ----------------------------------------------
  cohort = (
      cohort.sort_values(["subject_id", "intime"])
            .groupby("subject_id", as_index=False)
            .first()
  )

  print(f"-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: {len(cohort):,} ca")

  # Tính thời gian nằm ICU (tính theo ngày)
  cohort["icu_los_days"] = (
      pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
  ).dt.total_seconds() / 86400
  cohort["icu_los_hours"] = cohort["icu_los_days"] * 24

  # ---------------------------------------------
  # Tiêu chí 3: ICU stay trên 1 ngày (>= 1 ngày, tức >= 24 giờ)
  # ---------------------------------------------
  cohort = cohort[cohort["icu_los_days"] >= 1.0]
  print(f"-> Sau khi lọc ICU stay > 1 ngày: {len(cohort):,} ca")

  # Tạo mốc thời gian kết thúc cửa sổ quan sát (intime + 24h)
  cohort['obs_end_time'] = cohort['intime'] + pd.Timedelta(hours=24)

  print("=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===")
  """
  Tiêu chí loại bỏ các lượt ICU vi phạm logic:
  - Tiêu chí 1: Các khóa phải là duy nhất
  - Tiêu chí 2: Kiểm tra thứ tự các môc thời gian logic
    - Thời gian nhập viện <= Thời gian vào ICU < Thời gian vào ICU + 24h <= Thời gian ra ICU <= Thời gian ra viện
  - Tiêu chí 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
    - Thời điểm tử vong của bệnh nhân phải lớn hơn thười gian vào ICU + 24h

  """

  # List lưu giữ các stay_id vi phạm cần loại bỏ
  invalid_stay_ids = set()

  # Check 1: Kiểm tra tính duy nhất của Khóa
  total_rows = len(cohort)
  unique_stays = cohort['stay_id'].nunique()
  print(f"Tổng số dòng: {total_rows} | Số stay_id duy nhất: {unique_stays}")
  if total_rows != unique_stays:
      print("Bị trùng lặp stay_id do phép Join! Cần loại bỏ bản ghi trùng.")
      cohort = cohort.drop_duplicates(subset=['stay_id'])

  # Check 2: Kiểm tra thứ tự mốc thời gian logic
  invalid_time_mask = (
        (cohort['admittime'] > cohort['intime']) |
        (cohort['intime'] >= cohort['outtime']) |
        (cohort['outtime'] > cohort['dischtime'])
    )
  time_faulty_ids = cohort[invalid_time_mask]['stay_id'].tolist()
  invalid_stay_ids.update(time_faulty_ids)
  print(f"Phát hiện {len(time_faulty_ids)} lượt ICU vi phạm thứ tự thời gian sinh lý.")

  # Check 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
  early_death_mask = (
        (cohort['hospital_expire_flag'] == 1) &
        (cohort['deathtime'].notna()) &
        (cohort['deathtime'] <= cohort['obs_end_time'])
    )
  early_death_ids = cohort[early_death_mask]['stay_id'].tolist()
  invalid_stay_ids.update(early_death_ids)
  print(f"Phát hiện {len(early_death_ids)} bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.")

  # Loai bỏ các bản ghi vi phạm khỏi Cohort chính thức
  clean_cohort = cohort[~cohort['stay_id'].isin(invalid_stay_ids)].reset_index(drop=True)

  print("=== BẮT ĐẦU THỰC HIỆN TIỀN XỬ LÝ ===")
  """
  Các bước thực hiện:
  - Loại bỏ các cột không dùng đến
  - Chuyển đổi dạng dữ liệu
  - Đổi tên cột
  """
  clean_cohort["Outcome"]   = clean_cohort["hospital_expire_flag"].astype(int) # Chuyển biến outcome về dạng nhị phân (0/1)
  clean_cohort["Sex"]       = (clean_cohort["gender"] == "M").astype(int) # Chuyển biến giới tính về dạng nhị phân (0/1)
  clean_cohort["LOS_hours"] = clean_cohort["los"].astype(float) * 24 # Tạo biến los - giờ nằm icu

  clean_cohort = clean_cohort.rename(columns={
          "subject_id": "PatientID", "hadm_id": "AdmissionID", # Đổi mã bệnh nhân,  mã lần nhập viện
          "stay_id": "StayID",       "intime":  "ICUInTime", # Đổi mã nằm icu, thời gian vào ICU
          "outtime": "ICUOutTime",   "anchor_age": "Age", # Đổi thời gian ra icu, tuổi bệnh nhân
          "admission_type": "AdmissionType"
      })[["PatientID", "AdmissionID", "StayID",
          "ICUInTime", "ICUOutTime",
          "Age", "Sex", "AdmissionType", "Outcome", "LOS_hours"]].reset_index(drop=True) # Lấy các đặc trưng cần lấy

  print(f"=== KHỞI TẠO COHORT THÀNH CÔNG ===")
  print(f"Số lượng bệnh nhân hợp lệ cuối cùng: {len(clean_cohort)}")
  print(f"Thời gian nằm ICU trung bình: {cohort['icu_los_days'].mean():.2f} ngày")
  print(f"Tỷ lệ tử vong (Mortality Rate): {clean_cohort['Outcome'].mean():.2%}")

  return clean_cohort


In [8]:
print("Loading core tables...")
patients   = load("hosp", "patients.parquet")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.parquet")    # Thông tin nhập viện
icustays   = load("icu",  "icustays.parquet")      # Thông tin lần nằm

cohort = build_and_val_icu_cohort(patients, admissions, icustays)

Loading core tables...
  ✓ hosp/patients.parquet: 364,627 rows × 3 cols
  ✓ hosp/admissions.parquet: 546,028 rows × 16 cols
  ✓ icu/icustays.parquet: 94,458 rows × 8 cols
Tổng số lượt ICU ban đầu: 94,458
-> Sau khi lọc người trưởng thành (>=18t): 94,458 ca
-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: 65,366 ca
-> Sau khi lọc ICU stay > 1 ngày: 51,839 ca
=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===
Tổng số dòng: 51839 | Số stay_id duy nhất: 51839
Phát hiện 8714 lượt ICU vi phạm thứ tự thời gian sinh lý.
Phát hiện 171 bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.
=== BẮT ĐẦU THỰC HIỆN TIỀN XỬ LÝ ===
=== KHỞI TẠO COHORT THÀNH CÔNG ===
Số lượng bệnh nhân hợp lệ cuối cùng: 43124
Thời gian nằm ICU trung bình: 4.26 ngày
Tỷ lệ tử vong (Mortality Rate): 4.39%


In [9]:
cohort.isna().sum()

,0
PatientID,0
AdmissionID,0
StayID,0
ICUInTime,0
ICUOutTime,0
Age,0
Sex,0
AdmissionType,0
Outcome,0
LOS_hours,0


In [10]:
cohort.columns

Index(['PatientID', 'AdmissionID', 'StayID', 'ICUInTime', 'ICUOutTime', 'Age',
       'Sex', 'AdmissionType', 'Outcome', 'LOS_hours'],
      dtype='object')

In [132]:
cohort.head()

,PatientID,AdmissionID,StayID,ICUInTime,ICUOutTime,Age,Sex,AdmissionType,Outcome,LOS_hours
0,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,86,0,EW EMER.,0,93.438056
1,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,55,0,EW EMER.,0,26.832778
2,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-12 23:59:56,46,0,EW EMER.,0,32.126111
3,10002013,23581541,39060235,2160-05-18 10:00:53,2160-05-19 17:33:33,53,0,SURGICAL SAME DAY ADMISSION,0,31.544444
4,10002114,27793700,34672098,2162-02-17 23:30:00,2162-02-20 21:16:27,56,1,OBSERVATION ADMIT,0,69.774167


### **2.2. Chia tập train - test - validate để huấn luyện mô hình & Xác định nhãn cho bài toán**

- Thực hiện chia tập dữ liệu với tỷ lệ tử vong ở mỗi tập là như nhau
  - Chia theo tý lệ train (70%) - test (20%) - val (10%)
- **Nhãn mục tiêu** `hospital_expire_flag`
  - Với `0` là bệnh nhân sống sót và `1` là bệnh nhân tử vong

In [11]:
# Sinh x và y
X = cohort.drop(columns=['Outcome'])
y = cohort['Outcome']

In [12]:
from sklearn.model_selection import train_test_split

def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    target_col: str = 'hospital_expire_flag',
    test_size: float = 0.2,
    val_size: float = 0.125,
    random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Chia dataset thành 3 tập Train/Val/Test theo `subject_id` để chống Data Leakage.
    Tỷ lệ mặc định: 70% Train - 20% Val - 10% Test.
    """
    # Tách 20% cho tập Test (Còn lại 80% cho Train + Val)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # Chia 80% đó thành Train (70% tổng) và Val (10% tổng)
    # Tỷ lệ tập Val trong tập Train_Val là: 10% / 80% = 0.125 (12.5%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=val_size,
        stratify=y_train_val,
        random_state=random_state
    )

    # 4. Kiểm tra kích thước các tập
    print(f"Kích thước tập Train: {X_train.shape[0]} ({len(X_train)/len(cohort):.0%})")
    print(f"Kích thước tập Val:   {X_val.shape[0]} ({len(X_val)/len(cohort):.0%})")
    print(f"Kích thước tập Test:  {X_test.shape[0]} ({len(X_test)/len(cohort):.0%})")

    # 5. Kiểm tra tỷ lệ biến mục tiêu có được giữ nguyên không
    print("\nTỷ lệ nhãn trong tập Train:\n", y_train.value_counts(normalize=True))
    print("\nTỷ lệ nhãn trong tập Val:\n", y_val.value_counts(normalize=True))
    print("\nTỷ lệ nhãn trong tập Test:\n", y_test.value_counts(normalize=True))

    return X_train, X_test, y_train_val, y_test, X_train, X_val, y_train, y_val

In [13]:
X_train, X_test, y_train_val, y_test, X_train, X_val, y_train, y_val = split_data(X, y)

Kích thước tập Train: 30186 (70%)
Kích thước tập Val:   4313 (10%)
Kích thước tập Test:  8625 (20%)

Tỷ lệ nhãn trong tập Train:
 Outcome
0    0.956105
1    0.043895
Name: proportion, dtype: float64

Tỷ lệ nhãn trong tập Val:
 Outcome
0    0.956179
1    0.043821
Name: proportion, dtype: float64

Tỷ lệ nhãn trong tập Test:
 Outcome
0    0.956174
1    0.043826
Name: proportion, dtype: float64


In [14]:
X_train.shape, X_test.shape, X_val.shape, y_train.shape, y_test.shape, y_val.shape

((30186, 9), (8625, 9), (4313, 9), (30186,), (8625,), (4313,))

In [137]:
y_train.value_counts()

,count
Outcome,
0,28861
1,1325


## **3. Xây dựng các bảng đặc trưng thực hiện tiền xử lý cho dữ liệu huấn luyện**

In [15]:
from typing import Dict, Tuple, List
import shutil

In [16]:
VALID_RANGE = {
    "HR": (10, 300), "RR": (2, 80), "Temp_C": (25, 45), "Temp_F": (77, 113), "SBP": (30, 300), "DBP": (10, 200), "MBP": (20, 250), "SpO2": (50, 100),
    "GCS_total": (3, 15), "WBC": (0.1, 500), "HCT": (5, 70), "PLT": (1, 3000), "pH": (6.5, 8.5), "FiO2": (0.1, 1.05), "PaO2": (10, 700), "PaCO2": (10, 200),
    "HCO3": (5, 60), "CKMB": (0, 10000), "BNP": (1, 50000), "AST": (1, 50000), "ALT": (1, 50000), "Bili_total": (0.1, 100), "Bili_direct": (0.1, 100),
    "DDimer": (0.1, 100000), "BUN": (1, 500), "Creatinine": (0.1, 50), "Cortisol": (0.1, 200), "Na": (100, 180), "K": (1, 10), "Cl": (70, 150),
    "Mg": (0.5, 10), "Glucose": (10, 2000), "Height": (50, 250), "Weight": (10, 400), "WBC_lab": (0.1, 500), "HCT_lab": (5, 70), "PLT_lab": (1, 3000),
    "pH_lab": (6.5, 8.5), "PaO2_lab": (10, 700), "PaCO2_lab": (10, 200), "HCO3_lab": (5, 60), "Creatinine_lab": (0.1, 50), "BUN_lab": (1, 500),
    "Albumin_lab": (0.5, 8), "Lactate_lab": (0.1, 30), "CRP_lab": (0, 500), "Ca_lab": (3, 20), "Na_lab": (100, 180), "K_lab": (1, 10), "Cl_lab": (70, 150), "Mg_lab": (0.5, 10),
}

### **3.1. Xây dựng bảng chart vitals từ `chartevent`**

Kết hợp với bảng cohort với cửa số được lấy trong vòng 24h khi nằm ở icu
- intime <= charttime <= intime + 24h
- Lọc các chỉ số theo mã id hợp lệ rồi học các giá trị ngoại lai
- Thực hiện sinh thêm các đặc trưng mới với các chỉ số phức tạp
- Chuyển đổi dơn vị về chuẩn chung

In [17]:
CHART_ITEM2VAR = {
    220045: "HR", 220210: "RR", 224690: "RR", 224422: "RR",
    223761: "Temp_F", 223762: "Temp_C",
    220179: "SBP", 220050: "SBP", 220180: "DBP", 220051: "DBP", 220052: "MBP", 220181: "MBP",
    220277: "SpO2",
    220739: "GCS_eye", 223901: "GCS_motor", 223900: "GCS_verbal",
    223835: "FiO2",
    220621: "Glucose",
    226512: "Weight_kg", 226531: "Weight_lbs"
}

GCS_EYE_MAP = {"None": 0, "1 No Response": 1, "2 To pain": 2, "To Pain": 2, "3 To speech": 3, "To Speech": 3, "4 Spontaneously": 4, "Spontaneously": 4}
GCS_MOTOR_MAP = {"1 No Response": 1, "No response": 1, "2 Abnorm extensn": 2, "Abnormal extension": 2, "3 Abnorm flexion": 3, "Abnormal Flexion": 3, "4 Flex-withdraws": 4, "Flex-withdraws": 4, "5 Localizes Pain": 5, "Localizes Pain": 5, "6 Obeys Commands": 6, "Obeys Commands": 6}
GCS_VERBAL_MAP = {"No Response-ETT": 1, "No Response": 1, "1 No Response": 1, "1.0 ET/Trach": 1, "2 Incomp sounds": 2, "Incomprehensible sounds": 2, "3 Inapprop words": 3, "Inappropriate Words": 3, "4 Confused": 4, "Confused": 4, "5 Oriented": 5, "Oriented": 5}

CHART_FEATURES = ["HR", "RR", "Temp_C", "SBP", "DBP", "MBP", "SpO2", "GCS_total", "FiO2", "Glucose", "Weight"]

In [18]:
def process_chartevents(data_dir, cohort, hours=24, output_path=None, chunksize=1_000_000):
    print(f"Đang xử lý chartevents (Parquet) theo chunk (chunksize={chunksize})...")

    # Chuẩn bị dict tra cứu
    stay2intime = cohort.set_index("StayID")["ICUInTime"].to_dict()
    valid_stays = set(cohort["StayID"].tolist())
    valid_items = set(CHART_ITEM2VAR.keys())

    file_path = os.path.join(data_dir, "icu", "chartevents.parquet")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    temp_dir = os.path.join(data_dir, "temp_chartevents_chunks")
    os.makedirs(temp_dir, exist_ok=True)
    chunk_files = []

    try:
        # Đọc Parquet theo dạng Stream (Tối ưu RAM < 1GB)
        parquet_file = pq.ParquetFile(file_path)
        cols = ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "value", "valuenum"]

        for i, batch in enumerate(parquet_file.iter_batches(batch_size=chunksize, columns=cols)):
            chunk = batch.to_pandas()

            # Lọc sơ bộ stay_id và itemid ngay khi nạp chunk
            chunk = chunk[chunk["stay_id"].isin(valid_stays) & chunk["itemid"].isin(valid_items)]
            if chunk.empty:
                continue

            # Tính toán thời gian & Lọc số giờ nhập ICU
            chunk["icu_intime"] = chunk["stay_id"].map(stay2intime)
            chunk["charttime"] = pd.to_datetime(chunk["charttime"])
            chunk["hours_in"] = (chunk["charttime"] - chunk["icu_intime"]).dt.total_seconds() / 3600
            chunk = chunk[(chunk["hours_in"] >= 0) & (chunk["hours_in"] < hours)].copy()

            if chunk.empty:
                continue

            chunk["Variable"] = chunk["itemid"].map(CHART_ITEM2VAR)
            chunk["hour_bucket"] = chunk["hours_in"].astype(int).clip(0, hours - 1)

            # Chuyển đổi đơn vị
            m = chunk["Variable"] == "Temp_F"
            if m.any():
                chunk.loc[m, "valuenum"] = (pd.to_numeric(chunk.loc[m, "valuenum"], errors="coerce") - 32) * 5.0 / 9.0
                chunk.loc[m, "Variable"] = "Temp_C"

            m = chunk["Variable"] == "Weight_lbs"
            if m.any():
                chunk.loc[m, "valuenum"] = pd.to_numeric(chunk.loc[m, "valuenum"], errors="coerce") * 0.453592
                chunk.loc[m, "Variable"] = "Weight"
            chunk.loc[chunk["Variable"] == "Weight_kg", "Variable"] = "Weight"

            m = chunk["Variable"] == "Height_in"
            if m.any():
                chunk.loc[m, "valuenum"] = pd.to_numeric(chunk.loc[m, "valuenum"], errors="coerce") * 2.54
                chunk.loc[m, "Variable"] = "Height"
            chunk.loc[chunk["Variable"] == "Height_cm", "Variable"] = "Height"

            # Ánh xạ điểm GCS thành phần
            for var, mapping in [("GCS_eye", GCS_EYE_MAP), ("GCS_motor", GCS_MOTOR_MAP), ("GCS_verbal", GCS_VERBAL_MAP)]:
                m = chunk["Variable"] == var
                if m.any():
                    chunk.loc[m, "valuenum"] = chunk.loc[m, "value"].map(mapping)

            # Ép kiểu số & Lọc dữ liệu trống
            chunk["valuenum"] = pd.to_numeric(chunk["valuenum"], errors="coerce")
            chunk = chunk.dropna(subset=["valuenum"])

            # Lọc Outliers
            if 'VALID_RANGE' in globals() and VALID_RANGE is not None:
                for feat, (lo, hi) in VALID_RANGE.items():
                    m = chunk["Variable"] == feat
                    if m.any():
                        chunk = chunk[~(m & ((chunk["valuenum"] < lo) | (chunk["valuenum"] > hi)))]

            if chunk.empty:
                continue

            # Gom nhóm cục bộ (Local Aggregation)
            local_agg = (
                chunk.groupby(["subject_id", "hadm_id", "hour_bucket", "Variable"])["valuenum"]
                .agg(val_sum="sum", val_count="count")
                .reset_index()
            )

            # Ghi file đệm Parquet siêu nhẹ
            chunk_file = os.path.join(temp_dir, f"chunk_{i}.parquet")
            local_agg.to_parquet(chunk_file, index=False)
            chunk_files.append(chunk_file)

        if not chunk_files:
            print("Không tìm thấy dữ liệu phù hợp!")
            return pd.DataFrame(columns=["PatientID", "AdmissionID", "hour_bucket"] + CHART_FEATURES)

        # Tổng hợp toàn cục từ các file đệm
        all_chunks = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)

        global_agg = (
            all_chunks.groupby(["subject_id", "hadm_id", "hour_bucket", "Variable"])
            .agg(total_sum=("val_sum", "sum"), total_count=("val_count", "sum"))
            .reset_index()
        )
        global_agg["valuenum"] = global_agg["total_sum"] / global_agg["total_count"]

        # Pivot sang dạng Wide Format
        wide = global_agg.pivot_table(
            index=["subject_id", "hadm_id", "hour_bucket"],
            columns="Variable",
            values="valuenum"
        ).reset_index()

        wide.columns.name = None
        wide = wide.rename(columns={"subject_id": "PatientID", "hadm_id": "AdmissionID"})

        # Tính toán GCS_total từ các điểm thành phần nếu cột này nằm trong CHART_FEATURES
        if "GCS_total" in CHART_FEATURES and "GCS_total" not in wide.columns:
            gcs_cols = [c for c in ["GCS_eye", "GCS_motor", "GCS_verbal"] if c in wide.columns]
            if gcs_cols:
                wide["GCS_total"] = wide[gcs_cols].sum(axis=1, min_count=1)

        # Bổ sung các cột thiếu bằng NaN
        for col in CHART_FEATURES:
            if col not in wide.columns:
                wide[col] = np.nan

        result = wide[["PatientID", "AdmissionID", "hour_bucket"] + CHART_FEATURES]

        if output_path:
            result.to_parquet(output_path, index=False)
            print(f"Đã ghi kết quả xuống đĩa thành công: {output_path}")

        return result

    finally:
        # Tự động dọn dẹp các file rác trung gian
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

In [19]:
chart = process_chartevents(DATA_DIR, X_train, hours=24, output_path=None, chunksize=1_000_000)

Đang xử lý chartevents (Parquet) theo chunk (chunksize=1000000)...


In [20]:
chart.head()

,PatientID,AdmissionID,hour_bucket,HR,RR,Temp_C,SBP,DBP,MBP,SpO2,GCS_total,FiO2,Glucose,Weight
0,10000690,25860671,0,78.0,24.333333,36.500000,106.0,56.5,67.5,98.0,15.0,NaN,NaN,55.156787
1,10000690,25860671,1,71.0,26.000000,NaN,99.0,49.0,61.0,98.0,NaN,NaN,NaN,NaN
2,10000690,25860671,2,62.0,20.000000,NaN,95.0,38.0,53.0,99.0,NaN,NaN,NaN,NaN
3,10000690,25860671,3,62.0,19.000000,NaN,107.0,41.0,73.0,99.0,NaN,NaN,NaN,NaN
4,10000690,25860671,4,73.0,21.000000,35.555556,116.0,44.0,61.0,93.0,NaN,NaN,NaN,NaN


### **3.2. Xây dựng bảng lab test từ bảng `labevents`**

In [21]:
LAB_ITEM2VAR = {
    # Huyết học & Đông máu
    51301: "WBC_lab", 51221: "HCT_lab", 51265: "PLT_lab", 51274: "PT_lab", 51275: "PTT_lab",
    # Khí máu & Toan-Kiềm (APS III Acid-Base & LODS)
    50820: "pH_lab", 50821: "PaO2_lab", 50818: "PaCO2_lab", 50882: "HCO3_lab", 50813: "Lactate_lab",
    50816: "FiO2_lab",
    # Sinh hóa, Thận, Gan & Đường huyết
    51006: "BUN_lab", 50912: "Creatinine_lab", 50862: "Albumin_lab", 50885: "Bili_total_lab",
    50931: "Glucose_lab", # BỔ SUNG: Glucose xét nghiệm
    # Điện giải
    50983: "Na_lab", 50971: "K_lab"
}

LAB_FEATURES = [
    "WBC_lab", "HCT_lab", "PLT_lab", "PT_lab", "PTT_lab",
    "pH_lab", "PaO2_lab", "PaCO2_lab", "HCO3_lab", "Lactate_lab", "FiO2_lab",
    "BUN_lab", "Creatinine_lab", "Albumin_lab", "Bili_total_lab", "Glucose_lab",
    "Na_lab", "K_lab"
]

In [22]:
def process_labevents(data_dir, cohort, hours=24, output_path=None, chunksize=1_000_000):
    print(f"Đang xử lý labevents (Parquet) theo chunk (chunksize={chunksize})...")

    # Chuẩn bị bảng tra cứu thông tin lượt vào ICU
    cohort_lookup = cohort[["StayID", "AdmissionID", "ICUInTime"]].copy()
    cohort_lookup["ICUInTime"] = pd.to_datetime(cohort_lookup["ICUInTime"])

    valid_hadms = set(cohort_lookup["AdmissionID"].dropna().unique())
    valid_items = set(LAB_ITEM2VAR.keys())

    file_path = os.path.join(data_dir, "hosp", "labevents.parquet")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    temp_dir = os.path.join(data_dir, "temp_labevents_chunks")
    os.makedirs(temp_dir, exist_ok=True)
    chunk_files = []

    try:
        parquet_file = pq.ParquetFile(file_path)
        cols = ["subject_id", "hadm_id", "charttime", "itemid", "value", "valuenum"]

        for i, batch in enumerate(parquet_file.iter_batches(batch_size=chunksize, columns=cols)):
            chunk = batch.to_pandas()

            # Lọc sơ bộ theo hadm_id và itemid
            chunk = chunk[chunk["hadm_id"].isin(valid_hadms) & chunk["itemid"].isin(valid_items)].copy()
            if chunk.empty:
                continue

            # Merge với cohort_lookup để lấy thông tin ICUInTime và StayID chuẩn xác
            chunk = chunk.merge(cohort_lookup, left_on="hadm_id", right_on="AdmissionID", how="inner")

            # Tính toán thời gian & Lọc theo số giờ chỉ định (0 -> hours)
            chunk["charttime"] = pd.to_datetime(chunk["charttime"])
            chunk["hours_in"] = (chunk["charttime"] - chunk["ICUInTime"]).dt.total_seconds() / 3600
            chunk = chunk[(chunk["hours_in"] >= 0) & (chunk["hours_in"] < hours)].copy()

            if chunk.empty:
                continue

            chunk["Variable"] = chunk["itemid"].map(LAB_ITEM2VAR)
            chunk["hour_bucket"] = chunk["hours_in"].astype(int).clip(0, hours - 1)

            # Ép kiểu số & Lọc dữ liệu trống
            chunk["valuenum"] = pd.to_numeric(chunk["valuenum"], errors="coerce")
            chunk = chunk.dropna(subset=["valuenum"])

            # Chuẩn hóa FiO2_lab về dạng thập phân (0.21 - 1.0)
            m_fio2 = chunk["Variable"] == "FiO2_lab"
            if m_fio2.any():
                chunk.loc[m_fio2 & (chunk["valuenum"] > 1.0), "valuenum"] /= 100.0

            # Lọc Outliers
            if 'VALID_RANGE' in globals() and VALID_RANGE is not None:
                for feat, (lo, hi) in VALID_RANGE.items():
                    m = chunk["Variable"] == feat
                    if m.any():
                        chunk = chunk[~(m & ((chunk["valuenum"] < lo) | (chunk["valuenum"] > hi)))]

            if chunk.empty:
                continue

            # Gom nhóm cục bộ
            local_agg = (
                chunk.groupby(["subject_id", "hadm_id", "StayID", "hour_bucket", "Variable"])["valuenum"]
                .agg(val_sum="sum", val_count="count")
                .reset_index()
            )

            chunk_file = os.path.join(temp_dir, f"chunk_{i}.parquet")
            local_agg.to_parquet(chunk_file, index=False)
            chunk_files.append(chunk_file)

        if not chunk_files:
            print("Không tìm thấy dữ liệu phù hợp!")
            return pd.DataFrame(columns=["PatientID", "AdmissionID", "StayID", "hour_bucket"] + LAB_FEATURES)

        # Tổng hợp toàn cục từ các file đệm
        all_chunks = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)

        global_agg = (
            all_chunks.groupby(["subject_id", "hadm_id", "StayID", "hour_bucket", "Variable"])
            .agg(total_sum=("val_sum", "sum"), total_count=("val_count", "sum"))
            .reset_index()
        )
        global_agg["valuenum"] = global_agg["total_sum"] / global_agg["total_count"]

        # Pivot sang dạng Wide Format
        wide = global_agg.pivot_table(
            index=["subject_id", "hadm_id", "StayID", "hour_bucket"],
            columns="Variable",
            values="valuenum"
        ).reset_index()

        wide.columns.name = None
        wide = wide.rename(columns={"subject_id": "PatientID", "hadm_id": "AdmissionID"})

        # Bổ sung các cột thiếu bằng NaN
        for col in LAB_FEATURES:
            if col not in wide.columns:
                wide[col] = np.nan

        result = wide[["PatientID", "AdmissionID", "StayID", "hour_bucket"] + LAB_FEATURES]

        if output_path:
            result.to_parquet(output_path, index=False)
            print(f"Đã ghi kết quả xuống đĩa thành công: {output_path}")

        return result

    finally:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

In [23]:
lab = process_labevents(DATA_DIR, X_train, hours=24, output_path=None, chunksize=1_000_000)

Đang xử lý labevents (Parquet) theo chunk (chunksize=1000000)...


In [24]:
lab.head()

,PatientID,AdmissionID,StayID,hour_bucket,WBC_lab,HCT_lab,PLT_lab,PT_lab,PTT_lab,pH_lab,...,HCO3_lab,Lactate_lab,FiO2_lab,BUN_lab,Creatinine_lab,Albumin_lab,Bili_total_lab,Glucose_lab,Na_lab,K_lab
0,10000690,25860671.0,37081114,7,7.5,28.5,199.0,NaN,NaN,NaN,...,26.0,NaN,NaN,21.0,0.9,NaN,NaN,77.0,137.0,4.4
1,10001217,24597018.0,37067082,7,19.0,33.6,285.0,12.6,33.1,NaN,...,23.0,NaN,NaN,9.0,0.4,NaN,NaN,113.0,138.0,3.6
2,10002013,23581541.0,39060235,0,NaN,NaN,NaN,NaN,NaN,7.35,...,NaN,3.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10002013,23581541.0,39060235,1,NaN,NaN,NaN,NaN,NaN,7.40,...,NaN,2.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10002013,23581541.0,39060235,2,NaN,NaN,NaN,NaN,NaN,7.27,...,NaN,3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **3.3. Xây dựng các bảng còn lại phụ trợ cho bảng chính**

In [25]:
VASOPRESSOR_ITEMS = {221289: "Epinephrine", 221906: "Norepinephrine", 221653: "Dobutamine", 221662: "Dopamine"}
URINE_ITEMS = [226559, 226560, 226561, 226563, 226564, 226565, 226567, 226631, 226632, 227489]
CRRT_ITEM = 225802
VENT_ITEM = 225792

TREATMENT_FEATURES = [
    "Epinephrine_used", "Norepinephrine_used", "Dobutamine_used", "Dopamine_used",
    "Urine_output_mL", "CRRT_active", "Ventilated"
]

In [26]:
def process_vasopressors(data_dir, cohort, hours=24, output_path=None, chunksize=1_000_000):
    print(f"Xử lý Vasopressors (Parquet) theo chunk (chunksize={chunksize})...")

    stay2intime = cohort.set_index("StayID")["ICUInTime"].to_dict()
    stay2adm    = cohort.set_index("StayID")["AdmissionID"].to_dict()
    valid_stays = set(cohort["StayID"].tolist())
    vaso_cols   = [v + "_used" for v in VASOPRESSOR_ITEMS.values()]

    file_path = os.path.join(data_dir, "icu", "inputevents.parquet")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    temp_dir = os.path.join(data_dir, "temp_vaso_chunks")
    os.makedirs(temp_dir, exist_ok=True)
    chunk_files = []

    try:
        parquet_file = pq.ParquetFile(file_path)
        cols = ["stay_id", "starttime", "endtime", "itemid"]

        for i, batch in enumerate(parquet_file.iter_batches(batch_size=chunksize, columns=cols)):
            chunk = batch.to_pandas()

            # Lọc theo stay_id và itemid
            chunk = chunk[chunk["stay_id"].isin(valid_stays) & chunk["itemid"].isin(VASOPRESSOR_ITEMS)].copy()
            if chunk.empty:
                continue

            chunk["icu_intime"] = chunk["stay_id"].map(stay2intime)
            chunk = chunk.dropna(subset=["icu_intime"])

            chunk["starttime"] = pd.to_datetime(chunk["starttime"])
            chunk["endtime"]   = pd.to_datetime(chunk["endtime"])

            # Tính toán khoảng giờ bắt đầu và kết thúc
            chunk["start_h"] = ((chunk["starttime"] - chunk["icu_intime"]).dt.total_seconds() / 3600).clip(0, hours - 1).astype(int)
            chunk["end_h"]   = ((chunk["endtime"] - chunk["icu_intime"]).dt.total_seconds() / 3600).clip(0, hours - 1).astype(int)
            chunk["VarName"] = chunk["itemid"].map(VASOPRESSOR_ITEMS) + "_used"

            # Unroll khoảng thời gian ra từng hour_bucket
            rows = []
            for _, r in chunk.iterrows():
                adm_id = stay2adm.get(r["stay_id"])
                if adm_id is None:
                    continue
                for h in range(int(r["start_h"]), min(int(r["end_h"]) + 1, hours)):
                    rows.append({"AdmissionID": adm_id, "StayID": r["stay_id"], "hour_bucket": h, "VarName": r["VarName"], "val": 1})

            if not rows:
                continue

            # Nén dữ liệu theo từng chunk
            local_agg = (
                pd.DataFrame(rows)
                .groupby(["AdmissionID", "StayID", "hour_bucket", "VarName"])["val"]
                .max()
                .reset_index()
            )

            chunk_file = os.path.join(temp_dir, f"chunk_{i}.parquet")
            local_agg.to_parquet(chunk_file, index=False)
            chunk_files.append(chunk_file)

        if not chunk_files:
            print("Không tìm thấy dữ liệu Vasopressors phù hợp!")
            empty_df = pd.DataFrame(columns=["AdmissionID", "StayID", "hour_bucket"] + vaso_cols)
            return empty_df

        # Tổng hợp toàn cục từ các file đệm
        all_chunks = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
        global_agg = all_chunks.groupby(["AdmissionID", "StayID", "hour_bucket", "VarName"])["val"].max().reset_index()

        # Pivot Wide
        wide = global_agg.pivot_table(
            index=["AdmissionID", "StayID", "hour_bucket"],
            columns="VarName",
            values="val",
            fill_value=0
        ).reset_index()

        wide.columns.name = None

        # Bổ sung các cột thiếu bằng 0 (vì là biến nhị phân flag)
        for col in vaso_cols:
            if col not in wide.columns:
                wide[col] = 0

        result = wide[["AdmissionID", "StayID", "hour_bucket"] + vaso_cols]

        if output_path:
            result.to_parquet(output_path, index=False)
            print(f"Đã ghi Vasopressors xuống đĩa thành công: {output_path}")

        return result

    finally:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)


def process_interventions(data_dir, cohort, hours=24, output_path=None, chunksize=1_000_000):
    print(f"Xử lý Can thiệp (Urine + CRRT + Ventilation) theo chunk (chunksize={chunksize})...")

    stay2intime = cohort.set_index("StayID")["ICUInTime"].to_dict()
    stay2adm    = cohort.set_index("StayID")["AdmissionID"].to_dict()
    valid_stays = set(cohort["StayID"].tolist())

    # --- A. XỬ LÝ URINE OUTPUT (Bảng outputevents.parquet) ---
    print(" -> Đang đọc outputevents (Urine)...")
    urine_path = os.path.join(data_dir, "icu", "outputevents.parquet")
    temp_urine = os.path.join(data_dir, "temp_urine_chunks")
    os.makedirs(temp_urine, exist_ok=True)
    urine_chunks = []

    if os.path.exists(urine_path):
        pq_urine = pq.ParquetFile(urine_path)
        for i, batch in enumerate(pq_urine.iter_batches(batch_size=chunksize, columns=["stay_id", "charttime", "itemid", "value"])):
            chunk = batch.to_pandas()
            chunk = chunk[chunk["stay_id"].isin(valid_stays) & chunk["itemid"].isin(URINE_ITEMS)].copy()
            if chunk.empty:
                continue

            chunk["icu_intime"] = chunk["stay_id"].map(stay2intime)
            chunk["charttime"]  = pd.to_datetime(chunk["charttime"])
            chunk["hours_in"]   = (chunk["charttime"] - chunk["icu_intime"]).dt.total_seconds() / 3600
            chunk = chunk[(chunk["hours_in"] >= 0) & (chunk["hours_in"] < hours)].copy()

            if chunk.empty:
                continue

            chunk["hour_bucket"] = chunk["hours_in"].astype(int).clip(0, hours - 1)
            chunk["value"]       = pd.to_numeric(chunk["value"], errors="coerce")
            chunk                = chunk.dropna(subset=["value"])

            local_agg = (
                chunk.groupby(["stay_id", "hour_bucket"])["value"]
                .sum()
                .reset_index()
                .rename(columns={"value": "Urine_output_mL"})
            )

            f_path = os.path.join(temp_urine, f"u_chunk_{i}.parquet")
            local_agg.to_parquet(f_path, index=False)
            urine_chunks.append(f_path)

    # --- B. XỬ LÝ CRRT & VENTILATION (Bảng procedureevents.parquet) ---
    print(" -> Đang đọc procedureevents (CRRT + Vent)...")
    proc_path = os.path.join(data_dir, "icu", "procedureevents.parquet")
    temp_proc = os.path.join(data_dir, "temp_proc_chunks")
    os.makedirs(temp_proc, exist_ok=True)
    proc_chunks = []

    target_items = [CRRT_ITEM, VENT_ITEM]

    if os.path.exists(proc_path):
        pq_proc = pq.ParquetFile(proc_path)
        for i, batch in enumerate(pq_proc.iter_batches(batch_size=chunksize, columns=["stay_id", "starttime", "endtime", "itemid"])):
            chunk = batch.to_pandas()
            chunk = chunk[chunk["stay_id"].isin(valid_stays) & chunk["itemid"].isin(target_items)].copy()
            if chunk.empty:
                continue

            chunk["icu_intime"] = chunk["stay_id"].map(stay2intime)
            chunk = chunk.dropna(subset=["icu_intime"])

            chunk["starttime"] = pd.to_datetime(chunk["starttime"])
            chunk["endtime"]   = pd.to_datetime(chunk["endtime"])

            chunk["start_h"] = ((chunk["starttime"] - chunk["icu_intime"]).dt.total_seconds() / 3600).clip(0, hours - 1).astype(int)
            chunk["end_h"]   = ((chunk["endtime"] - chunk["icu_intime"]).dt.total_seconds() / 3600).clip(0, hours - 1).astype(int)

            rows = []
            for _, r in chunk.iterrows():
                var_name = "CRRT_active" if r["itemid"] == CRRT_ITEM else "Ventilated"
                for h in range(int(r["start_h"]), min(int(r["end_h"]) + 1, hours)):
                    rows.append({"stay_id": r["stay_id"], "hour_bucket": h, "VarName": var_name, "val": 1})

            if not rows:
                continue

            local_agg = pd.DataFrame(rows).groupby(["stay_id", "hour_bucket", "VarName"])["val"].max().reset_index()
            f_path = os.path.join(temp_proc, f"p_chunk_{i}.parquet")
            local_agg.to_parquet(f_path, index=False)
            proc_chunks.append(f_path)

    try:
        # Gộp dữ liệu Urine
        if urine_chunks:
            all_u = pd.concat([pd.read_parquet(f) for f in urine_chunks], ignore_index=True)
            u_df  = all_u.groupby(["stay_id", "hour_bucket"])["Urine_output_mL"].sum().reset_index()
        else:
            u_df = pd.DataFrame(columns=["stay_id", "hour_bucket", "Urine_output_mL"])

        # Gộp dữ liệu Procedure (CRRT + Vent)
        if proc_chunks:
            all_p = pd.concat([pd.read_parquet(f) for f in proc_chunks], ignore_index=True)
            p_df  = all_p.groupby(["stay_id", "hour_bucket", "VarName"])["val"].max().reset_index()
            p_wide = p_df.pivot_table(index=["stay_id", "hour_bucket"], columns="VarName", values="val", fill_value=0).reset_index()
            p_wide.columns.name = None
        else:
            p_wide = pd.DataFrame(columns=["stay_id", "hour_bucket", "CRRT_active", "Ventilated"])

        # Outer Merge kết quả
        result = pd.merge(u_df, p_wide, on=["stay_id", "hour_bucket"], how="outer")

        # Map lại thông tin ID
        result["AdmissionID"] = result["stay_id"].map(stay2adm)
        result = result.rename(columns={"stay_id": "StayID"})

        # Đảm bảo các cột bắt buộc luôn tồn tại
        if "Urine_output_mL" not in result.columns: result["Urine_output_mL"] = np.nan
        if "CRRT_active" not in result.columns:      result["CRRT_active"] = 0
        if "Ventilated" not in result.columns:       result["Ventilated"] = 0

        # Điền 0 cho các flag can thiệp bị NaN do Outer Join
        result["CRRT_active"] = result["CRRT_active"].fillna(0).astype(int)
        result["Ventilated"]  = result["Ventilated"].fillna(0).astype(int)

        target_cols = ["AdmissionID", "StayID", "hour_bucket", "Urine_output_mL", "CRRT_active", "Ventilated"]
        result = result[target_cols]

        if output_path:
            result.to_parquet(output_path, index=False)
            print(f"Đã ghi Interventions xuống đĩa thành công: {output_path}")

        return result

    finally:
        # Tự động dọn sạch thư mục tạm
        for d in [temp_urine, temp_proc]:
            if os.path.exists(d):
                shutil.rmtree(d)

In [27]:
vaso = process_vasopressors(DATA_DIR, X_train, hours=24, output_path=None, chunksize=1_000_000)
urine_crrt = process_interventions(DATA_DIR, X_train, hours=24, output_path=None, chunksize=1_000_000)
print("Done!")

Xử lý Vasopressors (Parquet) theo chunk (chunksize=1000000)...
Xử lý Can thiệp (Urine + CRRT + Ventilation) theo chunk (chunksize=1000000)...
 -> Đang đọc outputevents (Urine)...
 -> Đang đọc procedureevents (CRRT + Vent)...
Done!


## **4. Tiền xử lý dữ liệu**

In [35]:
print("Merge Data")

# Merge 4 nguồn
merged = chart.merge(lab, on=["AdmissionID", "hour_bucket"], how="outer") \
              .merge(vaso, on=["AdmissionID", "hour_bucket"], how="outer") \
              .merge(urine_crrt, on=["AdmissionID", "hour_bucket"], how="outer")

for col in ["Epinephrine_used", "Norepinephrine_used", "Dobutamine_used", "Dopamine_used", "CRRT_active"]:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0)

# Full time grid (24 giờ)
all_adm   = X_train["AdmissionID"].unique()
full_grid = pd.MultiIndex.from_product([all_adm, range(24)], names=["AdmissionID", "hour_bucket"]).to_frame(index=False)
df = full_grid.merge(merged, on=["AdmissionID", "hour_bucket"], how="left").sort_values(["AdmissionID", "hour_bucket"]).reset_index(drop=True)

Merge Data


In [36]:
df.head()

,AdmissionID,hour_bucket,PatientID_x,HR,RR,Temp_C,SBP,DBP,MBP,SpO2,...,K_lab,StayID_y,Epinephrine_used,Norepinephrine_used,Dobutamine_used,Dopamine_used,StayID,Urine_output_mL,CRRT_active,Ventilated
0,20000147,0,14990224.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN
1,20000147,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN
2,20000147,2,14990224.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN
3,20000147,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN
4,20000147,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN


In [38]:
from sklearn.preprocessing import StandardScaler

In [37]:
ALL_FEATURES = CHART_FEATURES + LAB_FEATURES + TREATMENT_FEATURES
print(f"Số đặc trưng tổng cộng: {len(ALL_FEATURES)}")

Số đặc trưng tổng cộng: 36


In [40]:
print("Missing Mask, Linear Interpolation, Median Fill & Scaling...")

feat_cols = ALL_FEATURES
mask_cols = [f"{col}_mask" for col in feat_cols]

# 1. Missing Mask
for col in feat_cols:
    df[f"{col}_mask"] = (~df[col].isna()).astype(np.float32)

# 2. Linear Interpolation per-patient
for col in feat_cols:
    df[col] = df.groupby("AdmissionID")[col].transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    df[col] = df.groupby("AdmissionID")[col].transform(lambda s: s.ffill().bfill())

train_mask = df["AdmissionID"].isin(train_adm)
val_mask   = df["AdmissionID"].isin(val_adm)
test_mask  = df["AdmissionID"].isin(test_adm)

# 3. Median Fill từ Train
train_median = df.loc[train_mask, feat_cols].median()
df.loc[train_mask, feat_cols] = df.loc[train_mask, feat_cols].fillna(train_median)
df.loc[val_mask,   feat_cols] = df.loc[val_mask,   feat_cols].fillna(train_median)
df.loc[test_mask,  feat_cols] = df.loc[test_mask,  feat_cols].fillna(train_median)

# 4. StandardScaler: fit TRÊN TRAIN
scaler = StandardScaler()
df.loc[train_mask, feat_cols] = scaler.fit_transform(df.loc[train_mask, feat_cols])
df.loc[val_mask,   feat_cols] = scaler.transform(df.loc[val_mask,   feat_cols])
df.loc[test_mask,  feat_cols] = scaler.transform(df.loc[test_mask,  feat_cols])

Missing Mask, Linear Interpolation, Median Fill & Scaling...


KeyboardInterrupt: 

## **5. Huấn luyện mô hình học máy**